In [1]:
import torch

### set up CUDA as device if available
if torch.cuda.is_available():
    print("GPU is available")
    device = torch.device("cuda")
    cuda_id = torch.cuda.current_device()
    print(f"ID of current CUDA device:{torch.cuda.current_device()}")
    print(f"Name of current CUDA device:{torch.cuda.get_device_name(cuda_id)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("GPU is not available, using CPU")
    device = torch.device("cpu")
print(f"device: {device}")

GPU is available
cuda
CUDA version: 12.4
ID of current CUDA device:0
Name of current CUDA device:NVIDIA H200


In [2]:
import numpy as np
import random
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm, trange
from IPython import display
from sklearn.metrics import average_precision_score
# from conf_and_plot import confusion_matrix_plots
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import KFold
from sklearn.model_selection import ParameterGrid
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import kruskal
import torch.nn.functional as F
from IPython.display import clear_output
import statistics

def seed_all(seed):
    if not seed:
        seed = 10

    print("[ Using Seed : ", seed, " ]")

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

seed_all(2025)

[ Using Seed :  2025  ]


In [3]:
scint_thresh = 0.1 # set the phase scintillation threshold

### following variable not being used anywhere
scint_outlier_thresh = 5. # set the value that determines phase scintillation outliers (these data samples will be removed)

processed_data_2015 = pd.read_csv("processed_data_2015.csv")
processed_data_2015 = processed_data_2015.drop(processed_data_2015.columns[0], axis=1)
predicted_label = 'sigmaPhi projected to vertical at prediction time [radians]'
y = processed_data_2015[predicted_label].values
print(y.shape)

X_fSelect = processed_data_2015.drop(predicted_label, axis=1)
X_fSelect = X_fSelect.values
print(X_fSelect.shape)

(4465846,)
(4465846, 15)


In [4]:
class simple_timeSeries_CNN(nn.Module):
    def __init__(self, cnn_filters, fc_size, dropout_p, loss, sequence_length):
        super(simple_timeSeries_CNN, self).__init__()
        
        self.loss = loss
        self.sequence_length = sequence_length

        self.conv1 = nn.Conv1d(in_channels=15, out_channels=cnn_filters, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(num_features=cnn_filters)
        self.dropout = nn.Dropout(dropout_p)
        self.conv2 = nn.Conv1d(in_channels=cnn_filters, out_channels=cnn_filters*2, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(num_features=cnn_filters*2)
        self.pool = nn.MaxPool1d(2)
        
        ### for simple, non true time-series CNN:
        # num_features = 25, MaxPool1d_size=2
        # 12 = round(25 / 2)
        # 6 = round(12 / 2)
        # self.fc1 = nn.Linear(cnn_filters * 2 * 6, fc_size)
        
        ### for true time-series CNN:
        # sequence_length, MaxPool1d_size=2
        # round(sequence_length / 2) = sequence_length // 2
        # per maxpool layer = sequence_length // 4
        
        self.fc1 = nn.Linear(cnn_filters * 2 * (sequence_length // 4), fc_size)
        # self.fc2 = nn.Linear(fc_size, 1)
        self.fc2 = nn.Linear(fc_size, sequence_length)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.dropout(x)
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1) # Flatten the output for the fully connected layers
        x = torch.relu(self.fc1(x))
        
        # second fully connected layer + reshape to [batch_size, sequence_length, 1]
        x = self.fc2(x).view(x.size(0), self.sequence_length, 1)

        if self.loss == 'focal':
            # x = torch.sigmoid(self.fc2(x))
            x = torch.sigmoid(x)
        elif self.loss == 'bce':
            # x = self.fc2(x)
            pass
        
        return x

In [5]:
def true_skill_score(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    tss = (TP / (TP + FN)) - (FP / (FP + TN))
    return tss

def heidke_skill_score(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    
    numerator = 2 * (TP * TN - FP * FN)
    denominator = (TP + FP) * (FP + TN) + (TP + FN) * (TN + FP)
    
    # Avoid division by zero
    if denominator == 0:
        return 0.0
    
    hss = numerator / denominator
    return hss

In [6]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction  # 'mean', 'sum', or 'none'

    def forward(self, inputs, targets):
        BCE_loss = F.binary_cross_entropy(inputs, targets, reduction='none')
        p_t = inputs * targets + (1 - inputs) * (1 - targets)
        focal_loss = self.alpha * (1 - p_t) ** self.gamma * BCE_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        elif self.reduction == 'none':
            return focal_loss
        else:
            raise ValueError(f"Invalid reduction method: {self.reduction}")

In [7]:
def train_model_with_cv(model, train_dataloader, val_dataloader, criterion, optimizer, scheduler, num_epochs):

    epochs = []
    training_loss = []
    validation_loss = []
    training_tss = []
    validation_tss = []
    training_hss = []
    validation_hss = []

    for epoch in range(num_epochs):

        ### training loop
        model.train()
        running_loss = 0.0
        predicted_training_labels = np.array([])
        y_train = np.array([])

        # for batch_inputs, batch_labels in tqdm(train_dataloader):
        for batch_inputs, batch_labels in train_dataloader:
            
            batch_inputs = batch_inputs.to(device)
            batch_labels = batch_labels.to(device)
            
            optimizer.zero_grad()
            output = model(batch_inputs)
            
            output.to(device)
            
            loss = criterion(output, batch_labels.float())
            loss.backward()
            optimizer.step()
            
            if scheduler == False:
                pass
            else:
                scheduler.step()
            
            running_loss += loss.item()
            with torch.no_grad(): predicted_training_labels = np.append(predicted_training_labels, output.cpu())
            with torch.no_grad(): y_train = np.append(y_train, batch_labels.cpu())

        predicted_training_labels = np.where(predicted_training_labels > 0.1, 1, 0)

        train_loss = running_loss / len(train_dataloader)
        train_tss = true_skill_score(y_train.astype(int), predicted_training_labels.astype(int))
        train_hss = heidke_skill_score(y_train.astype(int), predicted_training_labels.astype(int))

        ### validation loop
        model.eval()
        with torch.no_grad():
            running_loss = 0.0
            predicted_val_labels = np.array([])
            y_val = np.array([])

            # for batch_inputs, labels in tqdm(val_dataloader):
            for batch_inputs, batch_labels in val_dataloader:
            
                batch_inputs = batch_inputs.to(device)
                batch_labels = batch_labels.to(device)
                
                outputs = model(batch_inputs)
            
                outputs.to(device)
                
                outputs = model(batch_inputs)
                loss = criterion(outputs, batch_labels.float())
                running_loss += loss.item()
                predicted_val_labels = np.append(predicted_val_labels, outputs.cpu())
                y_val = np.append(y_val, batch_labels.cpu())

        predicted_val_labels = np.where(predicted_val_labels > 0.1, 1, 0)

        val_loss = running_loss / len(val_dataloader)
        val_tss = true_skill_score(y_val.astype(int), predicted_val_labels.astype(int))
        val_hss = heidke_skill_score(y_val.astype(int), predicted_val_labels.astype(int))

        epochs.append(epoch)
        training_loss.append(train_loss)
        validation_loss.append(val_loss)
        training_tss.append(train_tss)
        validation_tss.append(val_tss)
        training_hss.append(train_hss)
        validation_hss.append(val_hss)

    return training_hss, validation_hss, training_tss, validation_tss


In [8]:
class CNNBinaryTimeSeriesClassifier():
    def __init__(self, cnn_filters=8, fc_size=64, sequence_length=60, dropout_p=0.1, batch_size=128, 
                 optimizer_type = 'adam', lr=0.001, sgd_momentum=0.9, weight_decay=0.0001,
                 loss='bce', bce_pos_class_weight=50, focal_alpha=0.25, focal_gamma=2,
                 num_epochs=5, cv_folds=5, scheduler_t=0):
        self.cnn_filters = cnn_filters
        self.fc_size = fc_size
        self.sequence_length = sequence_length
        self.dropout_p = dropout_p
        self.batch_size = batch_size
        self.optimizer_type = optimizer_type
        self.lr = lr        
        self.weight_decay = weight_decay
        self.sgd_momentum = sgd_momentum
        self.loss = loss
        self.bce_pos_class_weight = bce_pos_class_weight
        self.focal_alpha = focal_alpha
        self.focal_gamma = focal_gamma
        self.num_epochs = num_epochs
        self.cv_folds = cv_folds
        self.scheduler_t = scheduler_t
        
    def reshape_data_to_seq_length(self, data, seq_len):
        num_samples = data.shape[0]
        input_features = data.shape[1]
        num_batches = num_samples // seq_len
        data = data[:num_batches * seq_len] # truncate points that don't divide evenly
        reshaped_data = data.reshape(num_batches, seq_len, input_features)
        return reshaped_data

    def reshape_labels_to_seq_length(self, labels, seq_len):
        labels = torch.from_numpy(labels) if isinstance(labels, np.ndarray) else labels
        num_samples = labels.shape[0]
        num_batches = num_samples // seq_len
        labels = labels[:num_batches * seq_len] # truncate points that don't divide evenly
        labels = labels.view(num_batches, seq_len, -1)
        return labels
    
    def fit(self, X, y):

        cv_tss_values = []
        cv_hss_values = []

        tscv = TimeSeriesSplit(n_splits=self.cv_folds)
        for i, (train_index, val_index) in enumerate(tscv.split(X)):
            
            model = simple_timeSeries_CNN(cnn_filters=self.cnn_filters, 
                                          fc_size=self.fc_size, 
                                          dropout_p=self.dropout_p, 
                                          loss=self.loss, 
                                          sequence_length=self.sequence_length)
            model.to(device)
            
            if self.loss == 'bce':
                criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([self.bce_pos_class_weight]).to(device))
            elif self.loss == 'focal':
                criterion = FocalLoss(alpha=self.focal_alpha, gamma=self.focal_gamma)

            if self.optimizer_type == 'adam':
                optimizer = optim.Adam(model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
            elif self.optimizer_type == 'sgd':
                optimizer = optim.SGD(model.parameters(), lr=self.lr, momentum=self.sgd_momentum, weight_decay=self.weight_decay)

            if self.scheduler_t > 0:
                scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.scheduler_t)
            else:
                scheduler = False

            X_train, X_val = X[train_index], X[val_index]
            y_train, y_val = y[train_index], y[val_index]
            
            scaler_X = RobustScaler()
            scaler_X = scaler_X.fit(X_train)

            X_train_scaled = scaler_X.transform(X_train)
            X_val_scaled = scaler_X.transform(X_val)
            
            X_train_scaled = self.reshape_data_to_seq_length(X_train_scaled, self.sequence_length)
            X_val_scaled = self.reshape_data_to_seq_length(X_val_scaled, self.sequence_length)

            train_labels = self.reshape_labels_to_seq_length(y_train, self.sequence_length)
            val_labels = self.reshape_labels_to_seq_length(y_val, self.sequence_length)
            
            train_inputs = torch.tensor(X_train_scaled, dtype=torch.float32).transpose(1, 2)  # Shape: [batch_size, 15, 1]
            train_inputs = train_inputs.to(device)
            train_labels = train_labels.to(device)
            train_dataset = TensorDataset(train_inputs, train_labels)
            train_dataloader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=False, worker_init_fn=seed_worker)

            val_inputs = torch.tensor(X_val_scaled, dtype=torch.float32).transpose(1, 2)  # Shape: [batch_size, 15, 1]
            val_inputs = val_inputs.to(device)
            val_labels = val_labels.to(device)
            val_dataset = TensorDataset(val_inputs, val_labels)
            val_dataloader = DataLoader(val_dataset, batch_size=self.batch_size, shuffle=False, worker_init_fn=seed_worker)
            
            training_hss, validation_hss, training_tss, validation_tss = \
            train_model_with_cv(model, 
                 train_dataloader, val_dataloader, 
                 criterion, optimizer, scheduler,
                 num_epochs=self.num_epochs)
            cv_tss_values.append(validation_tss[-1])
            cv_hss_values.append(validation_hss[-1])

            if i == 0:
                total_params = sum(p.numel() for p in model.parameters())
                print(f'Total number of parameters: {total_params}')
                
            print(f"Fold {i}: train tss = {training_tss[-1]:.4f} : val tss = {validation_tss[-1]:.4f} ::: train hss = {training_hss[-1]:.4f} : val hss = {validation_hss[-1]:.4f}")
        
        return cv_hss_values, cv_tss_values
    

In [9]:
training_data_size = 50000

X_train, X_test, \
    y_train, y_test, \
        idx_train, idx_test = train_test_split(X_fSelect, y, range(len(y)), train_size=training_data_size, shuffle=False)

# ### small test
# param_grid = {
#     'cnn_filters': [256],   
#     'fc_size': [256, 1048],  
#     'sequence_length': [100],   
#     'dropout_p': [0.6],  
#     'batch_size': [64],  
#     'lr': [0.03],  
#     'optimizer_type': ['sgd'],  
#     'weight_decay':[0.05],  
#     'loss': ['bce'],
#     'bce_pos_class_weight': [35],  
#     'num_epochs': [20],  
#     'cv_folds': [5], 
# }

### strategic hypertune iteration 1
param_grid = {
    'cnn_filters': [16, 32],
    'fc_size': [64, 1048],
    'sequence_length': [15, 45],
    'dropout_p': [0.1, 0.5],
    'batch_size': [32, 512],
    'lr': [0.0001, 0.01],
    'optimizer_type': ['adam', 'sgd'],
    'weight_decay':[0.0001, 0.01],
    'loss': ['bce'],
    'bce_pos_class_weight': [30, 75],
#     'scheduler_t': [5],
    'num_epochs': [50],
    'cv_folds': [5], 
}

### strategic hypertune iteration 2
param_grid = {
    'cnn_filters': [16, 64],
    'fc_size': [64, 1048],
    'sequence_length': [30, 45],  ### changed
    'dropout_p': [0.1, 0.5],
    'batch_size': [512, 1024],  ### changed
    'lr': [0.005, 0.03],  ### changed
    'optimizer_type': ['adam', 'sgd'],
    'weight_decay':[0.001, 0.01],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [30, 75],
#     'scheduler_t': [5],
    'num_epochs': [50],
    'cv_folds': [5], 
}

### strategic hypertune iteration 3
param_grid = {
    'cnn_filters': [8, 16],  ### changed
    'fc_size': [128, 1536],  ### changed
    'sequence_length': [40, 45],  ### changed
    'dropout_p': [0.1, 0.5],
    'batch_size': [768, 1536],  ### changed
    'lr': [0.005, 0.01],  ### changed
    'optimizer_type': ['adam', 'sgd'],
    'weight_decay':[0.001, 0.01],  
    'loss': ['bce'],
    'bce_pos_class_weight': [40, 65],  ### changed
    'num_epochs': [50],
    'cv_folds': [5], 
}

### strategic hypertune iteration 4
param_grid = {
    'cnn_filters': [4, 8, 128],  ### changed, added 1 dimension
    'fc_size': [512, 2048],  ### changed
    'sequence_length': [15, 45],  ### changed
    'dropout_p': [0.1, 0.5],
    'batch_size': [1024, 2048],  ### changed
    'lr': [0.0005, 0.01],  ### changed
    'optimizer_type': ['adam', 'sgd'],
    'weight_decay':[0.005, 0.05],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [40, 65],  
    'num_epochs': [50],
    'cv_folds': [5], 
}

### strategic hypertune iteration 5
param_grid = {
    'cnn_filters': [8, 64, 128],  ### changed
    'fc_size': [512, 2048],  
    'sequence_length': [15, 45],  
    'dropout_p': [0.1, 0.5],
    'batch_size': [128, 256],  ### changed
    'lr': [0.001, 0.025],  ### changed
    'optimizer_type': ['adam', 'sgd'],
    'weight_decay':[0.005, 0.05],  
    'loss': ['bce'],
    'bce_pos_class_weight': [25, 50],  ### changed
    'num_epochs': [50],
    'cv_folds': [5], 
}

### strategic hypertune iteration 6
param_grid = {
    'cnn_filters': [4, 8, 32],  ### changed
    'fc_size': [512, 1024],  ### changed
    'sequence_length': [30, 45],  ### changed
    'dropout_p': [0.1, 0.5],
    'batch_size': [128, 256],  
    'lr': [0.001, 0.025],  
    'optimizer_type': ['adam', 'sgd'],
    'weight_decay':[0.005, 0.075],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [30, 60],  ### changed
    'num_epochs': [50],
    'cv_folds': [5], 
}

### strategic hypertune iteration 7
param_grid = {
    'cnn_filters': [4, 8, 16],  ### changed
    'fc_size': [512, 768],  ### changed
    'sequence_length': [15, 45],  ### changed
    'dropout_p': [0.1, 0.5],
    'batch_size': [128, 256],  
    'lr': [0.005, 0.025],  ### changed
    'optimizer_type': ['adam', 'sgd'],
    'weight_decay':[0.01, 0.05],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [45, 60],  ### changed
    'num_epochs': [50],
    'cv_folds': [5], 
}

### strategic hypertune iteration 8
param_grid = {
    'cnn_filters': [8, 32, 64],  ### changed 
    'fc_size': [1024, 2560],  ### changed
    'sequence_length': [15, 45],  
    'dropout_p': [0.1, 0.25],  ### changed
    'batch_size': [128, 256],  
    'lr': [0.01, 0.025],  ### changed
    'optimizer_type': ['adam', 'sgd'],
    'weight_decay':[0.05, 0.075],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [35, 55],  ### changed
    'num_epochs': [50],
    'cv_folds': [5], 
}

### strategic hypertune iteration 9
param_grid = {
    'cnn_filters': [4, 8, 24],  ### changed
    'fc_size': [1024, 2560],  
    'sequence_length': [20, 45],  ### changed
    'dropout_p': [0.1, 0.35],  ### changed
    'batch_size': [128, 256],  
    'lr': [0.0075, 0.015],  ### changed
    'optimizer_type': ['adam', 'sgd'],
    'weight_decay':[0.01, 0.025],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [30, 60],  ### changed
    'num_epochs': [50],
    'cv_folds': [5], 
}

### strategic hypertune iteration 10
param_grid = {
    'cnn_filters': [4, 8, 12],  ### changed
    'fc_size': [768, 1024],  ### changed
    'sequence_length': [20, 45],  
    'dropout_p': [0.05, 0.2],  ### changed
    'batch_size': [128, 256],  
    'lr': [0.0075, 0.015],  
    'optimizer_type': ['adam', 'sgd'],
    'weight_decay':[0.025, 0.06],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [40, 70],  ### changed
    'num_epochs': [50],
    'cv_folds': [5], 
}

param_combinations = list(ParameterGrid(param_grid))

grid = pd.DataFrame()
tss_distributions = []
hss_distributions = []
mean_tss_scores = []
mean_hss_scores = []
for params in param_combinations:

    print(str(param_combinations.index(params)) + "/" + str(len(param_combinations)))
    print(params)

    cnn_time_series_classifier = CNNBinaryTimeSeriesClassifier(**params)
    cv_hss_values, cv_tss_values = cnn_time_series_classifier.fit(X_train, y_train)
    
    tss_distributions.append(cv_tss_values)
    hss_distributions.append(cv_hss_values)
    mean_cv_tss = sum(cv_tss_values)/len(cv_tss_values)
    mean_cv_hss = sum(cv_hss_values)/len(cv_hss_values)
    print(f"mean cv TSS: {mean_cv_tss:.4f}")
    print(f"mean cv HSS: {mean_cv_hss:.4f}")
    mean_tss_scores.append(mean_cv_tss)
    mean_hss_scores.append(mean_cv_hss)
    
    if params == param_combinations[0]:
        grid = pd.DataFrame(params, index=[0])
    else:
        grid = pd.concat([grid, pd.DataFrame([params])], ignore_index=True)

grid['tss'] = [round(num, 4) for num in mean_tss_scores]
grid['hss'] = [round(num, 4) for num in mean_hss_scores]
grid = grid.sort_values(by=['tss', 'hss'], ascending=False)
grid = grid.drop(columns=[col for col in ['num_epochs','cv_folds','loss','scheduler_t'] if col in grid.columns], axis=1)
grid.head(20)


0/768
{'batch_size': 128, 'bce_pos_class_weight': 40, 'cnn_filters': 4, 'cv_folds': 5, 'dropout_p': 0.05, 'fc_size': 768, 'loss': 'bce', 'lr': 0.0075, 'num_epochs': 50, 'optimizer_type': 'adam', 'sequence_length': 20, 'weight_decay': 0.025}
Total number of parameters: 47180
Fold 0: train tss = 0.6905 : val tss = -0.0261 ::: train hss = 0.2444 : val hss = -0.0116
Fold 1: train tss = 0.8327 : val tss = 0.1079 ::: train hss = 0.3659 : val hss = 0.0598
Fold 2: train tss = 0.6800 : val tss = 0.7256 ::: train hss = 0.2165 : val hss = 0.4379
Fold 3: train tss = 0.7608 : val tss = 0.1900 ::: train hss = 0.2491 : val hss = 0.0383
Fold 4: train tss = 0.7345 : val tss = 0.6415 ::: train hss = 0.2284 : val hss = 0.2074
mean cv TSS: 0.3278
mean cv HSS: 0.1464
1/768
{'batch_size': 128, 'bce_pos_class_weight': 40, 'cnn_filters': 4, 'cv_folds': 5, 'dropout_p': 0.05, 'fc_size': 768, 'loss': 'bce', 'lr': 0.0075, 'num_epochs': 50, 'optimizer_type': 'adam', 'sequence_length': 20, 'weight_decay': 0.06}
Tot

Total number of parameters: 103269
Fold 0: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
Fold 1: train tss = 0.7830 : val tss = 0.2612 ::: train hss = 0.3766 : val hss = 0.1124
Fold 2: train tss = 0.6800 : val tss = 0.7611 ::: train hss = 0.2166 : val hss = 0.4180
Fold 3: train tss = 0.6606 : val tss = 0.5710 ::: train hss = 0.1668 : val hss = 0.0737
Fold 4: train tss = 0.6767 : val tss = 0.6475 ::: train hss = 0.1873 : val hss = 0.2520
mean cv TSS: 0.4482
mean cv HSS: 0.1712
12/768
{'batch_size': 128, 'bce_pos_class_weight': 40, 'cnn_filters': 4, 'cv_folds': 5, 'dropout_p': 0.05, 'fc_size': 768, 'loss': 'bce', 'lr': 0.015, 'num_epochs': 50, 'optimizer_type': 'sgd', 'sequence_length': 20, 'weight_decay': 0.025}
Total number of parameters: 47180
Fold 0: train tss = 0.3982 : val tss = 0.0028 ::: train hss = 0.3704 : val hss = 0.0050
Fold 1: train tss = 0.8014 : val tss = 0.2429 ::: train hss = 0.4618 : val hss = 0.2289
Fold 2: train tss = 0.6831 : val ts

Total number of parameters: 137573
Fold 0: train tss = 0.0500 : val tss = 0.0000 ::: train hss = 0.0952 : val hss = 0.0000
Fold 1: train tss = 0.8116 : val tss = 0.2831 ::: train hss = 0.4783 : val hss = 0.1097
Fold 2: train tss = 0.7480 : val tss = 0.6980 ::: train hss = 0.1929 : val hss = 0.3273
Fold 3: train tss = 0.7316 : val tss = 0.2950 ::: train hss = 0.2784 : val hss = 0.0858
Fold 4: train tss = 0.7580 : val tss = 0.6803 ::: train hss = 0.3046 : val hss = 0.2763
mean cv TSS: 0.3913
mean cv HSS: 0.1598
23/768
{'batch_size': 128, 'bce_pos_class_weight': 40, 'cnn_filters': 4, 'cv_folds': 5, 'dropout_p': 0.05, 'fc_size': 1024, 'loss': 'bce', 'lr': 0.0075, 'num_epochs': 50, 'optimizer_type': 'sgd', 'sequence_length': 45, 'weight_decay': 0.06}
Total number of parameters: 137573
Fold 0: train tss = 0.2498 : val tss = 0.0092 ::: train hss = 0.3700 : val hss = 0.0177
Fold 1: train tss = 0.7644 : val tss = 0.3594 ::: train hss = 0.3989 : val hss = 0.1904
Fold 2: train tss = 0.6846 : val 

Total number of parameters: 47180
Fold 0: train tss = 0.0500 : val tss = -0.0001 ::: train hss = 0.0952 : val hss = -0.0002
Fold 1: train tss = 0.7285 : val tss = 0.4239 ::: train hss = 0.2657 : val hss = 0.2039
Fold 2: train tss = 0.4735 : val tss = 0.4466 ::: train hss = 0.1867 : val hss = 0.3159
Fold 3: train tss = 0.6774 : val tss = 0.2959 ::: train hss = 0.2467 : val hss = 0.0726
Fold 4: train tss = 0.6809 : val tss = 0.4791 ::: train hss = 0.2424 : val hss = 0.2206
mean cv TSS: 0.3291
mean cv HSS: 0.1625
34/768
{'batch_size': 128, 'bce_pos_class_weight': 40, 'cnn_filters': 4, 'cv_folds': 5, 'dropout_p': 0.2, 'fc_size': 768, 'loss': 'bce', 'lr': 0.0075, 'num_epochs': 50, 'optimizer_type': 'adam', 'sequence_length': 45, 'weight_decay': 0.025}
Total number of parameters: 103269
Fold 0: train tss = 0.7939 : val tss = -0.0153 ::: train hss = 0.3650 : val hss = -0.0189
Fold 1: train tss = 0.8230 : val tss = 0.2955 ::: train hss = 0.4210 : val hss = 0.1062
Fold 2: train tss = 0.7686 : v

Total number of parameters: 47180
Fold 0: train tss = 0.0983 : val tss = -0.0043 ::: train hss = 0.1092 : val hss = -0.0062
Fold 1: train tss = 0.7507 : val tss = 0.1898 ::: train hss = 0.3922 : val hss = 0.1450
Fold 2: train tss = 0.6330 : val tss = 0.6422 ::: train hss = 0.1876 : val hss = 0.2498
Fold 3: train tss = 0.7412 : val tss = 0.2578 ::: train hss = 0.2913 : val hss = 0.0397
Fold 4: train tss = 0.6521 : val tss = 0.6754 ::: train hss = 0.1686 : val hss = 0.2501
mean cv TSS: 0.3522
mean cv HSS: 0.1357
45/768
{'batch_size': 128, 'bce_pos_class_weight': 40, 'cnn_filters': 4, 'cv_folds': 5, 'dropout_p': 0.2, 'fc_size': 768, 'loss': 'bce', 'lr': 0.015, 'num_epochs': 50, 'optimizer_type': 'sgd', 'sequence_length': 20, 'weight_decay': 0.06}
Total number of parameters: 47180
Fold 0: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
Fold 1: train tss = 0.6911 : val tss = 0.3867 ::: train hss = 0.1435 : val hss = 0.2047
Fold 2: train tss = 0.5069 : val tss

...

Total number of parameters: 318997
Fold 0: train tss = 0.2494 : val tss = -0.0018 ::: train hss = 0.3325 : val hss = -0.0034
Fold 1: train tss = 0.7953 : val tss = 0.3282 ::: train hss = 0.2667 : val hss = 0.1230
Fold 2: train tss = 0.7049 : val tss = 0.6562 ::: train hss = 0.1152 : val hss = 0.2390
Fold 3: train tss = 0.7463 : val tss = 0.2319 ::: train hss = 0.1944 : val hss = 0.0427
Fold 4: train tss = 0.7512 : val tss = 0.7379 ::: train hss = 0.1858 : val hss = 0.3300
mean cv TSS: 0.3905
mean cv HSS: 0.1462
727/768
{'batch_size': 256, 'bce_pos_class_weight': 70, 'cnn_filters': 12, 'cv_folds': 5, 'dropout_p': 0.05, 'fc_size': 1024, 'loss': 'bce', 'lr': 0.0075, 'num_epochs': 50, 'optimizer_type': 'sgd', 'sequence_length': 45, 'weight_decay': 0.06}
Total number of parameters: 318997
Fold 0: train tss = 0.3484 : val tss = 0.0020 ::: train hss = 0.3484 : val hss = 0.0036
Fold 1: train tss = 0.7528 : val tss = 0.3183 ::: train hss = 0.2233 : val hss = 0.1138
Fold 2: train tss = 0.6123 : 

Total number of parameters: 109820
Fold 0: train tss = 0.9631 : val tss = 0.1629 ::: train hss = 0.1113 : val hss = 0.1452
Fold 1: train tss = 0.8119 : val tss = 0.2653 ::: train hss = 0.1493 : val hss = 0.0550
Fold 2: train tss = 0.6657 : val tss = 0.6017 ::: train hss = 0.1063 : val hss = 0.1910
Fold 3: train tss = 0.7269 : val tss = 0.2069 ::: train hss = 0.1711 : val hss = 0.0566
Fold 4: train tss = 0.6804 : val tss = 0.6530 ::: train hss = 0.1399 : val hss = 0.2341
mean cv TSS: 0.3780
mean cv HSS: 0.1364
738/768
{'batch_size': 256, 'bce_pos_class_weight': 70, 'cnn_filters': 12, 'cv_folds': 5, 'dropout_p': 0.2, 'fc_size': 768, 'loss': 'bce', 'lr': 0.0075, 'num_epochs': 50, 'optimizer_type': 'adam', 'sequence_length': 45, 'weight_decay': 0.025}
Total number of parameters: 239637
Fold 0: train tss = 0.9943 : val tss = 0.0971 ::: train hss = 0.4572 : val hss = 0.1069
Fold 1: train tss = 0.9483 : val tss = 0.2197 ::: train hss = 0.3936 : val hss = 0.0908
Fold 2: train tss = 0.8524 : va

Total number of parameters: 109820
Fold 0: train tss = 0.5441 : val tss = -0.0008 ::: train hss = 0.2720 : val hss = -0.0010
Fold 1: train tss = 0.8122 : val tss = 0.1635 ::: train hss = 0.2087 : val hss = 0.0726
Fold 2: train tss = 0.7944 : val tss = 0.7308 ::: train hss = 0.2253 : val hss = 0.4148
Fold 3: train tss = 0.7780 : val tss = 0.0736 ::: train hss = 0.2237 : val hss = 0.0259
Fold 4: train tss = 0.7682 : val tss = 0.5780 ::: train hss = 0.2053 : val hss = 0.1540
mean cv TSS: 0.3090
mean cv HSS: 0.1333
749/768
{'batch_size': 256, 'bce_pos_class_weight': 70, 'cnn_filters': 12, 'cv_folds': 5, 'dropout_p': 0.2, 'fc_size': 768, 'loss': 'bce', 'lr': 0.015, 'num_epochs': 50, 'optimizer_type': 'sgd', 'sequence_length': 20, 'weight_decay': 0.06}
Total number of parameters: 109820
Fold 0: train tss = 0.0998 : val tss = 0.0000 ::: train hss = 0.1663 : val hss = 0.0000
Fold 1: train tss = 0.8168 : val tss = 0.1785 ::: train hss = 0.2418 : val hss = 0.0799
Fold 2: train tss = 0.6360 : val

Total number of parameters: 318997
Fold 0: train tss = 0.2984 : val tss = -0.0043 ::: train hss = 0.3061 : val hss = -0.0073
Fold 1: train tss = 0.7405 : val tss = 0.3845 ::: train hss = 0.1777 : val hss = 0.1169
Fold 2: train tss = 0.6870 : val tss = 0.6495 ::: train hss = 0.0964 : val hss = 0.2329
Fold 3: train tss = 0.6547 : val tss = 0.4422 ::: train hss = 0.1278 : val hss = 0.0522
Fold 4: train tss = 0.6789 : val tss = 0.6395 ::: train hss = 0.1334 : val hss = 0.2122
mean cv TSS: 0.4223
mean cv HSS: 0.1214
760/768
{'batch_size': 256, 'bce_pos_class_weight': 70, 'cnn_filters': 12, 'cv_folds': 5, 'dropout_p': 0.2, 'fc_size': 1024, 'loss': 'bce', 'lr': 0.015, 'num_epochs': 50, 'optimizer_type': 'adam', 'sequence_length': 20, 'weight_decay': 0.025}
Total number of parameters: 145916
Fold 0: train tss = 0.6784 : val tss = 0.1788 ::: train hss = 0.1274 : val hss = 0.1775
Fold 1: train tss = 0.8796 : val tss = 0.2547 ::: train hss = 0.2331 : val hss = 0.0889
Fold 2: train tss = 0.7036 : 

,batch_size,bce_pos_class_weight,cnn_filters,dropout_p,fc_size,lr,optimizer_type,sequence_length,weight_decay,tss,hss
241,128,70,4,0.20,1024,0.0075,adam,20,0.060,0.5183,0.1725
669,256,70,8,0.05,1024,0.0150,sgd,20,0.060,0.4704,0.1828
78,128,40,8,0.05,768,0.0150,sgd,45,0.025,0.4702,0.1542
15,128,40,4,0.05,768,0.0150,sgd,45,0.060,0.4677,0.1785
16,128,40,4,0.05,1024,0.0075,adam,20,0.025,0.4640,0.2074
172,128,40,12,0.20,768,0.0150,sgd,20,0.025,0.4638,0.1887
200,128,70,4,0.05,768,0.0150,adam,20,0.025,0.4624,0.1802
291,128,70,8,0.20,768,0.0075,adam,45,0.060,0.4611,0.1597
289,128,70,8,0.20,768,0.0075,adam,20,0.060,0.4575,0.1616
729,256,70,12,0.05,1024,0.0150,adam,20,0.060,0.4546,0.1399


In [10]:
grid.sort_values(by=['hss', 'tss'], ascending=False).head(20)

,batch_size,bce_pos_class_weight,cnn_filters,dropout_p,fc_size,lr,optimizer_type,sequence_length,weight_decay,tss,hss
64,128,40,8,0.05,768,0.0075,adam,20,0.025,0.4194,0.2422
176,128,40,12,0.20,1024,0.0075,adam,20,0.025,0.3852,0.2390
456,256,40,8,0.05,768,0.0150,adam,20,0.025,0.4427,0.2247
514,256,40,12,0.05,768,0.0075,adam,45,0.025,0.3886,0.2243
440,256,40,4,0.20,1024,0.0150,adam,20,0.025,0.3835,0.2183
82,128,40,8,0.05,1024,0.0075,adam,45,0.025,0.3800,0.2137
144,128,40,12,0.05,1024,0.0075,adam,20,0.025,0.3961,0.2134
53,128,40,4,0.20,1024,0.0075,sgd,20,0.060,0.4289,0.2129
560,256,40,12,0.20,1024,0.0075,adam,20,0.025,0.4337,0.2110
81,128,40,8,0.05,1024,0.0075,adam,20,0.060,0.3069,0.2107


In [11]:
if grid.empty:
    print("grid empty")
else:
    grid.to_csv("1_cnn_ts_hypertune_grid_output_10.csv")
    print("grid output successful, saved as 1_cnn_ts_hypertune_grid_output_10.csv")
    

grid output successful, saved as 1_cnn_ts_hypertune_grid_output_10.csv
